In [1]:
print("hi")

hi


In [2]:
import json
import re

with open("soil.json", "r", encoding="utf-8") as f:
    text = f.read()

# Fix soil_types issue
text = re.sub(r'"soil_types":\s*,', '"soil_types": [] ,', text)

data = json.loads(text)

print("Valid JSON loaded successfully")

Valid JSON loaded successfully


In [14]:
with open("soil_fixed.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

In [6]:
import geopandas as gpd


In [9]:
gdf = gpd.read_file("gadm41_IND_shp/gadm41_IND_2.shp")

mp = gdf[gdf["NAME_1"] == "Madhya Pradesh"]

mp = mp.to_crs(epsg=4326)

mp.to_file("mp_districts.geojson", driver="GeoJSON")

In [11]:
print(mp.geom_type.unique())
print(mp.columns)

['Polygon' 'MultiPolygon']
Index(['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2',
       'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2',
       'geometry'],
      dtype='object')


In [15]:
import json
import pandas as pd

with open("soil_fixed.json") as f:
    soil = json.load(f)

soil_df = pd.DataFrame(soil)

print(soil_df.columns)

Index(['district', 'state', 'soil_types', 'dominant_soil',
       'soil_characteristics', 'agricultural_suitability', 'geometry',
       'geometry_source', 'soil_data_source'],
      dtype='object')


In [16]:
mp["NAME_2"] = mp["NAME_2"].str.strip().str.lower()
soil_df["district"] = soil_df["district"].str.strip().str.lower()

In [17]:
mp_set = set(mp["NAME_2"])
soil_set = set(soil_df["district"])

print("In MP but not in soil:", mp_set - soil_set)
print("In soil but not in MP:", soil_set - mp_set)

In MP but not in soil: {'narsimhapur', 'east nimar', 'west nimar'}
In soil but not in MP: {'khargone', 'narsinghpur', 'khandwa', 'niwari'}


In [18]:
name_mapping = {
    "narsimhapur": "narsinghpur",
    "east nimar": "khandwa",
    "west nimar": "khargone"
}

mp["NAME_2"] = mp["NAME_2"].replace(name_mapping)

In [19]:
print(mp[mp["NAME_2"].str.contains("niwari")])

Empty GeoDataFrame
Columns: [GID_2, GID_0, COUNTRY, GID_1, NAME_1, NL_NAME_1, NAME_2, VARNAME_2, NL_NAME_2, TYPE_2, ENGTYPE_2, CC_2, HASC_2, geometry]
Index: []


In [20]:
mp_set = set(mp["NAME_2"])
soil_set = set(soil_df["district"])

print("Still missing:", mp_set - soil_set)
print("Extra soil entries:", soil_set - mp_set)

Still missing: set()
Extra soil entries: {'niwari'}


In [21]:
merged = mp.merge(
    soil_df,
    left_on="NAME_2",
    right_on="district",
    how="left"
)

print("Unmatched districts:", merged["dominant_soil"].isna().sum())

Unmatched districts: 0


In [24]:
print(merged.columns)

Index(['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2',
       'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2',
       'geometry_x', 'district', 'state', 'soil_types', 'dominant_soil',
       'soil_characteristics', 'agricultural_suitability', 'geometry_y',
       'geometry_source', 'soil_data_source'],
      dtype='object')


In [25]:
import geopandas as gpd

merged = gpd.GeoDataFrame(
    merged,
    geometry="geometry_x",
    crs="EPSG:4326"
)

In [26]:
if "geometry_y" in merged.columns:
    merged = merged.drop(columns=["geometry_y"])

In [27]:
from shapely.geometry import Point

merged.sindex  # build spatial index

def get_soil_info(lat, lon):
    point = Point(lon, lat)

    possible = merged.iloc[list(merged.sindex.intersection(point.bounds))]
    match = possible[possible.contains(point)]

    if not match.empty:
        row = match.iloc[0]
        return {
            "district": row["NAME_2"],
            "dominant_soil": row["dominant_soil"],
            "agricultural_suitability": row["agricultural_suitability"]
        }
    return {"error": "Outside Madhya Pradesh"}

In [28]:
print(get_soil_info(23.2599, 77.4126))  

{'district': 'bhopal', 'dominant_soil': 'Deep soil', 'agricultural_suitability': 'Soybean, Maize, Sorghum, Chickpea, Wheat'}
